# Lab C · Phân quyền và regression / Authorization and regression

Dùng hai người dùng giả có quyền đọc khác nhau. So sánh trước/sau trên cùng fixture, đồng thời kiểm tra truy cập bị chặn trước khi đọc dữ liệu. / Use two synthetic users with different read permissions. Compare before/after on the same fixture and check that access is denied before reading data.

**VI:** Chọn **Connect → Connect to a hosted runtime** (CPU là đủ), rồi chạy từng ô từ trên xuống. Không chọn *local runtime*. Lab dùng dữ liệu giả công khai; không cần khóa API hay kết nối Supabase. Bạn có thể lưu một bản notebook riêng rồi sửa mã để thử nghiệm. Colab có giới hạn tài nguyên và có thể xóa runtime khi không hoạt động, nên hãy tải báo cáo cuối bài.

**EN:** Connect to a **Google-hosted CPU runtime**, then run cells in order. Do not select a local runtime. The lab uses public synthetic data, without API keys or Supabase. Save a personal copy to edit and experiment. Colab has resource limits and may delete idle runtimes, so download the final report.

Nguồn / Sources: [Colab FAQ](https://research.google.com/colaboratory/faq.html). Ô thiết lập tải các module nhỏ từ một commit GitHub cố định cho mỗi lần chạy. / Setup downloads small modules from one GitHub commit per run. The public repository must contain this notebook and the lab files; before publication, the GitHub links will not work.


In [ ]:
# Only run in a Google-hosted Colab runtime, not a local runtime.
import importlib.util
import json
import re
import sys
import tempfile
from pathlib import Path, PurePosixPath
from urllib.request import Request, urlopen

try:
    in_colab = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    in_colab = False
if not in_colab or not Path("/content").is_dir():
    raise RuntimeError("Use a Google-hosted Colab runtime. Do not install on your PC.")

# Clear previous-run objects so a new source revision cannot reuse an old model/report.
for previous_state in ("backend", "report", "lab_a", "lab_b", "lab_c"):
    globals().pop(previous_state, None)

REPOSITORY = "Hzyl/JourneyAIEngineer"
REVISION = "main"  # Replace with a saved 40-character commit SHA to reproduce a run.

def fetch(url, limit=5_000_000):
    request = Request(url, headers={"User-Agent": "JourneyAIEngineer-Colab"})
    with urlopen(request, timeout=30) as response:
        payload = response.read(limit + 1)
    if len(payload) > limit:
        raise ValueError("Unexpectedly large download")
    return payload

if REVISION != "main" and not re.fullmatch(r"[0-9a-f]{40}", REVISION):
    raise ValueError("REVISION must be main or a full commit SHA")
try:
    COMMIT = (json.loads(fetch(f"https://api.github.com/repos/{REPOSITORY}/commits/main"))["sha"]
              if REVISION == "main" else REVISION)
    if not re.fullmatch(r"[0-9a-f]{40}", COMMIT):
        raise ValueError("Invalid commit returned by GitHub")
    tree = json.loads(fetch(f"https://api.github.com/repos/{REPOSITORY}/git/trees/{COMMIT}?recursive=1"))
    if tree.get("truncated"):
        raise ValueError("Incomplete GitHub tree")
    prefix = "samples/vi-document-assistant/"
    paths = [item["path"] for item in tree["tree"]
             if item["type"] == "blob" and item["path"].startswith(prefix + "research_lab/")
             and PurePosixPath(item["path"]).suffix in {".py", ".json"}]
    if not paths or len(paths) > 50:
        raise ValueError("Research labs are missing from this revision; publish them first.")
    LAB_ROOT = Path(tempfile.mkdtemp(prefix="journey-labs-"))
    for path in paths:
        relative = PurePosixPath(path.removeprefix(prefix))
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("Unsafe file path")
        target = LAB_ROOT.joinpath(*relative.parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(fetch(f"https://raw.githubusercontent.com/{REPOSITORY}/{COMMIT}/{path}"))
except Exception as error:
    raise RuntimeError("Could not load the published labs. Check the revision/network; "
                       "GitHub may rate-limit requests. Retry later. No model was loaded.") from error

# Rerunning this cell must not keep imports from an earlier repository revision.
for module in list(sys.modules):
    if module == "research_lab" or module.startswith("research_lab."):
        del sys.modules[module]
sys.path.insert(0, str(LAB_ROOT))
print("Repository commit:", COMMIT)
print("Files loaded:", len(paths))


In [ ]:
def show_report(report):
    """Display an overview; export still contains every case and provenance field."""
    print("Lab", report["lab"])
    if report["lab"] == "A":
        for method, result in report["methods"].items():
            for split in ("dev", "test"):
                if split not in result:
                    continue
                score = result[split]
                print(method, split, "cases =", score["count"])
                print("retrieval =", score["retrieval"])
                print("decision accuracy =", score["decision_accuracy"])
                print("failures =", score["failures"])
    elif report["lab"] == "B":
        for method, result in report["methods"].items():
            print(method, "success =", result["task_success_rate"],
                  "mean calls =", result["mean_calls"])
            for case in result["cases"]:
                print(" ", case["id"], case["stop_reason"], "calls =", case["calls"])
        print("checkpoint =", report["checkpoint"])
        print("failure probes =", [(case["id"], case["passed"])
                                   for case in report["failure_probes"]])
    else:
        for variant in ("before", "after"):
            result = report[variant]
            print(variant, result["passed"], "/", result["total"])
            print("failures =", [case["id"] for case in result["cases"] if not case["passed"]])
        print("regression =", report["regression"])
    print("Limits:", *report.get("limitations", []), sep="\n- ")


In [ ]:
from research_lab import lab_c
report = lab_c.run()
show_report(report)


## Đọc kết quả và thử thay đổi / Inspect and experiment

Tìm ca rò rỉ dữ liệu trong bản cố ý sai, rồi xem bản sửa chặn ở đâu. Đọc hash dữ liệu, cấu hình và các ca hồi quy. Fixture prompt injection chỉ kiểm tra ranh giới công cụ của mô phỏng; không chứng minh một LLM thật chống được mọi injection. / Identify leaks in the deliberately flawed baseline and where the corrected version blocks them. Compare fixture hashes, configuration and regression cases. Injection fixtures test mock tool boundaries, not general real-LLM resistance.

## Lưu bằng chứng / Save evidence

Tải báo cáo, ghi lại một lỗi bạn hiểu được và một thay đổi muốn thử. Tiến độ Journey cần được bạn tự cập nhật. / Download the report, explain one failure and propose one change. Update your Journey progress yourself.

In [ ]:
# Save the exact source revision and report; no Drive mount or API key needed.
from datetime import datetime, timezone
from google.colab import files

report["notebook_provenance"] = {
    "repository": REPOSITORY, "commit": COMMIT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "python": sys.version, "runtime": "Google Colab",
}
destination = LAB_ROOT / "lab-c-security-regression.json"
destination.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
files.download(str(destination))
